# Reddit Pain-Point Finder

This script executes functions at `/lib/reddit.py` to get a report of pain points given a subreddit.


## Setup


In [ ]:
import sys
sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv("../env")

from lib import reddit as Reddit

## Auth check


In [2]:
r = Reddit.get_client()
print(Reddit.check_auth(r))

Version 7.8.2 of praw is outdated. Version 8.0.3 was released Wednesday August 12, 2026.


{'read_only': True, 'subscribers': 86485, 'limits': {'remaining': 996.0, 'reset_timestamp': 1787453999.294997, 'used': 4}}


## Configure


In [8]:
IDEA_SLUG = "fitness-studio-ops"

# Replace with your actual vertical
SUBREDDITS = [
    "smallbusiness",
    "personaltraining",
    "crossfit",
    "no-shows",
    "late cancel",
    "waitlist",
    "instructor pay"
]

# Back-office / management pain
DOMAIN_NOUNS = [
    "class scheduling",
    "membership billing",
    "no-shows",
    "late cancel",
    "waitlist",
    "instructor pay",
]

# Community / engagement pain - your second hypothesis
COMMUNITY_NOUNS = [
    "member retention",
    "member engagement",
    "community app",
    "challenge tracking",
]

# The incumbents. Mindbody is the big one - widely complained about,
# which makes family 3 unusually productive in this vertical.
COMPETITORS = [
    "Mindbody",
    "Wodify",
    "PushPress",
    "Zen Planner",
    "Glofox",
    "WellnessLiving",
    "Vagaro",
    "ClassPass",
]

ALL_NOUNS = DOMAIN_NOUNS + COMMUNITY_NOUNS

# 1. Pain grammar x nouns - the complaint shape
QUERIES  = [f'"{p}" {n}' for p in Reddit.PAIN_PHRASES[:8] for n in ALL_NOUNS]

# 2. Workaround hunting - highest-value signal for vertical SaaS
QUERIES += [f'spreadsheet {n}' for n in ALL_NOUNS]
QUERIES += [f'manually {n}' for n in ALL_NOUNS]

# 3. Competitor complaints - people who already pay and still have the problem
QUERIES += [f'{c} {t}' for c in COMPETITORS
            for t in ["alternative", "switching from", "hate"]]

print(f"{len(QUERIES)} queries x {len(SUBREDDITS)} subs = {len(QUERIES)*len(SUBREDDITS)} searches")

124 queries x 7 subs = 868 searches


## Harvest


In [ ]:
cfg = Reddit.HarvestConfig(
    subreddits=SUBREDDITS[:2],
    queries=QUERIES,
    time_filters=["year"],
    sorts=["relevance"],
    limit_per_search=60,
    comments_per_post=40,
    expand_more_comments=0,
)

n_search = len(SUBREDDITS) * len(QUERIES)
print(f"~{n_search} searches, then ~1 call per submission for comments")

~868 searches, then ~1 call per submission for comments


## Full harvest


In [ ]:
import time

subs, comments = Reddit.harvest(r, cfg)

run_id = f"{IDEA_SLUG}-{int(time.time())}"
path = Reddit.save_json(subs + comments, f"../runs/{run_id}/raw.jsonl")
print(f"\n{len(subs)} submissions, {len(comments)} comments -> {path}")

   r/smallbusiness [relevance/year] '"there's no way to" no-shows' -> +1 new 1 raw
   r/smallbusiness [relevance/year] '"is there a way to" no-shows' -> +2 new 2 raw
   r/smallbusiness [relevance/year] '"is there a way to" member engagement' -> +1 new 1 raw
   r/smallbusiness [relevance/year] '"manually" class scheduling' -> +3 new 3 raw
   r/smallbusiness [relevance/year] '"manually" no-shows' -> +38 new 41 raw
   r/smallbusiness [relevance/year] '"manually" waitlist' -> +1 new 2 raw
   r/smallbusiness [relevance/year] '"manually" community app' -> +4 new 4 raw
   r/smallbusiness [relevance/year] '"manually" challenge tracking' -> +9 new 11 raw
   r/smallbusiness [relevance/year] '"by hand" no-shows' -> +5 new 6 raw
   r/smallbusiness [relevance/year] 'spreadsheet class scheduling' -> +56 new 60 raw
   r/smallbusiness [relevance/year] 'spreadsheet membership billing' -> +27 new 29 raw
   r/smallbusiness [relevance/year] 'spreadsheet no-shows' -> +12 new 19 raw
   r/smallbusiness [rele

In [ ]:
rows = Reddit.load_json(path)
df = pd.DataFrame(rows)
df["text"] = df["title"].fillna(df["body"].str.slice(0, 120))
df[["subreddit", "kind", "score", "text", "permalink"]].head(30)

,subreddit,kind,score,text,permalink
0,smallbusiness,submission,0,"The hidden cost of ""good enough"" inventory tra...",https://reddit.com/r/smallbusiness/comments/1t...
1,smallbusiness,submission,30,An angry ex-employee is ruining my hiring on G...,https://reddit.com/r/smallbusiness/comments/1u...
2,smallbusiness,submission,3,Need guidance for online consulting business,https://reddit.com/r/smallbusiness/comments/1t...
3,smallbusiness,submission,1,We're a small remote biz and our teams are get...,https://reddit.com/r/smallbusiness/comments/1o...
4,smallbusiness,submission,4,"How many appointment reminders is ""too many”?",https://reddit.com/r/smallbusiness/comments/1v...
5,smallbusiness,submission,5,How do small studios deal with last-minute can...,https://reddit.com/r/smallbusiness/comments/1u...
6,smallbusiness,submission,1,i charged 5k for this to a tutoring business a...,https://reddit.com/r/smallbusiness/comments/1t...
7,smallbusiness,submission,34,How do you reduce no-shows for service appoint...,https://reddit.com/r/smallbusiness/comments/1t...
8,smallbusiness,submission,0,How do you handle no-shows? Genuinely curious ...,https://reddit.com/r/smallbusiness/comments/1s...
9,smallbusiness,submission,1,Why I Won’t Use Xero Accounting Again After Th...,https://reddit.com/r/smallbusiness/comments/1t...


In [15]:
import pandas as pd
from lib import tokens as Tok

records = Reddit.load_json(path)
df = pd.DataFrame(records)
print(df.kind.value_counts())
print(df.subreddit.value_counts())
print("distinct authors:", df.author.nunique())

print("\nCORPUS")
for k, v in Tok.corpus_stats(records).items():
    print(f"  {k:16} {v}")

print("\nTRUNCATION CURVE")
print(pd.DataFrame(Tok.truncation_curve(records)))

print("\nCOST if you extracted everything:", Tok.estimate_cost(records))

kind
comment       8110
submission     717
Name: count, dtype: int64
subreddit
smallbusiness       6735
personaltraining    2092
Name: count, dtype: int64
distinct authors: 4147

CORPUS
  items            8827
  total_tokens     978438
  mean             110.8
  p50              63
  p90              275
  p99              730
  max              2933
  top1pct_share    0.106

TRUNCATION CURVE
  max_chars  total_tokens  pct_of_full  items_truncated  pct_items_truncated
0       400        483607         49.4             3477                 39.4
1       800        699872         71.5             1950                 22.1
2      1200        813250         83.1             1067                 12.1
3      2000        899654         91.9              239                  2.7
4      4000        951268         97.2               57                  0.6
5      none        978438        100.0                0                  0.0

COST if you extracted everything: {'items': 8827, 'batches': 110